### Импорты и загрузка

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import torch
from pathlib import Path

from dataset import build_item_vocab
from rm_index import RMIndex, RMDataset

PROC = Path('../data/processed')
train = pd.read_parquet(PROC / 'train.parquet')
val = pd.read_parquet(PROC / 'val.parquet')
test = pd.read_parquet(PROC / 'test.parquet')

# Объединённый датасет
all_df = pd.concat([train, val, test], ignore_index=True)
print(f"Всего: {len(all_df):,}")

item2idx, idx2item = build_item_vocab(train)

Всего: 4,632,197


### Индекс

In [2]:
idx = RMIndex(all_df, item2idx, t1='2021-07-24', t2='2021-08-04')

RMIndex: 4,632,197 событий
  train: 2,789,962
  val:   371,021
  test:  474,130


### Создание датасета, проверка примера

In [3]:
train_ds = RMDataset(idx, idx.train_mask, max_len=50)
val_ds = RMDataset(idx, idx.val_mask, max_len=50)
test_ds = RMDataset(idx, idx.test_mask, max_len=50)

input_ids, reward = train_ds[0]
print(f"input_ids.shape: {input_ids.shape}")
print(f"reward:          {reward.item():.4f}")

# Проверка: последний ненулевой токен — это кандидат
arr = input_ids.numpy()
non_pad = np.where(arr != 0)[0]
candidate_idx = arr[non_pad[-1]]
print(f"Кандидат index: {candidate_idx} -> item_id: {idx2item[candidate_idx]}")

# Сверивание с первым train-событием в индексе
first_row = idx.rows[0] if hasattr(idx, 'rows') else np.flatnonzero(idx.train_mask)[0]
print(f"Ожидалось item_id: {idx2item[idx.items[first_row]]}")
assert idx2item[candidate_idx] == idx2item[idx.items[first_row]]
print("\nКандидат совпадает с item в исходных данных")

RMDataset: 2,789,962 примеров
RMDataset: 371,021 примеров
RMDataset: 474,130 примеров
input_ids.shape: torch.Size([50])
reward:          0.0300
Кандидат index: 6394 -> item_id: 7102
Ожидалось item_id: 7102

Кандидат совпадает с item в исходных данных


### Сравнение с одним юзером

In [4]:
# Пример - интересный и чересчур активный найденный юзер 416206
sample_user = 416206
user_events = all_df[all_df['user_id'] == sample_user].sort_values('last_watch_dt')
print(f"Юзер {sample_user}: {len(user_events)} событий")

# Поиск событий этого юзера в индексе
user_rows = np.flatnonzero(idx.users == sample_user)
print(f"Событий в индексе: {len(user_rows)}")

# Первые 5 событий с историей
n_shown = 0
for i in user_rows:
    if not idx.has_history[i]:
        continue
    hi = idx.day_start[i]
    lo = idx.user_start[i]
    history_len = hi - lo
    print(f"  event {i}: history_len={history_len}, candidate={idx2item[idx.items[i]]}, "
          f"reward={idx.rewards[i]:.2f}, date={idx.dates[i]}")
    n_shown += 1
    if n_shown >= 5:
        break

Юзер 416206: 1285 событий
Событий в индексе: 1285
  event 1755922: history_len=4, candidate=6832, reward=0.00, date=2021-03-16T00:00:00.000000
  event 1755923: history_len=4, candidate=8566, reward=0.00, date=2021-03-16T00:00:00.000000
  event 1755924: history_len=4, candidate=3578, reward=0.00, date=2021-03-16T00:00:00.000000
  event 1755925: history_len=4, candidate=9481, reward=0.00, date=2021-03-16T00:00:00.000000
  event 1755926: history_len=4, candidate=14431, reward=0.13, date=2021-03-16T00:00:00.000000


### EDA награды

In [5]:
# Распределение reward по train
train_rewards = idx.rewards[idx.train_mask]
print("Распределение reward в train:")
print(pd.Series(train_rewards).describe(percentiles=[.1, .25, .5, .75, .9, .99]))

# Доля «нулевых» reward
print(f"\nReward = 0: {(train_rewards == 0).mean()*100:.1f}%")
print(f"Reward > 0.8: {(train_rewards > 0.8).mean()*100:.1f}%")

Распределение reward в train:
count    2.789962e+06
mean     5.268986e-01
std      4.268545e-01
min      0.000000e+00
10%      0.000000e+00
25%      7.000000e-02
50%      5.000000e-01
75%      1.000000e+00
90%      1.000000e+00
99%      1.000000e+00
max      1.000000e+00
dtype: float64

Reward = 0: 10.4%
Reward > 0.8: 42.3%


*Было обнаружено, что с датасете фигурируют юзеры, которые просматривают неприличное кол-во айтемов в день и показывают чрезвычайную активность (тот же юзер 416206). Такое бывает на таких датасетах, то нормально, но созаёт лишний шум. Нужно посмотреть, насколько их много и что с ними делать*

### Листающие юзеров

In [6]:
# Сколько событий в день у юзера
all_df['_date'] = all_df['last_watch_dt'].dt.normalize()
events_per_day = all_df.groupby(['user_id', '_date']).size().rename('n_events')

# Для каждого события в train - в каком дне он находится
train_rows = np.flatnonzero(idx.train_mask)
train_users_dates = pd.DataFrame({
    'user_id': idx.users[train_rows],
    'date': pd.to_datetime(idx.dates[train_rows]),
})
train_users_dates['n_events_in_day'] = train_users_dates.merge(
    events_per_day.reset_index(),
    left_on=['user_id', 'date'],
    right_on=['user_id', '_date'],
)['n_events'].values

print("Распределение числа событий в день для train-примеров:")
print(train_users_dates['n_events_in_day'].describe(percentiles=[.5, .75, .9, .95, .99]))

# Доля примеров из «дней с >30 событий»
n_heavy = (train_users_dates['n_events_in_day'] > 30).sum()
print(f"\nПримеров из дней с >30 событий: {n_heavy:,} ({n_heavy/len(train_rows)*100:.1f}%)")

# Средний reward из «тяжёлых» дней vs обычных
train_rewards_arr = idx.rewards[train_rows]
heavy_mask = train_users_dates['n_events_in_day'].values > 30
print(f"\nСредний reward:")
print(f"  обычные дни:   {train_rewards_arr[~heavy_mask].mean():.3f}")
print(f"  тяжёлые дни:   {train_rewards_arr[heavy_mask].mean():.3f}")

Распределение числа событий в день для train-примеров:
count    2.789962e+06
mean     2.719869e+00
std      4.678209e+00
min      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
90%      5.000000e+00
95%      7.000000e+00
99%      1.400000e+01
max      2.500000e+02
Name: n_events_in_day, dtype: float64

Примеров из дней с >30 событий: 5,941 (0.2%)

Средний reward:
  обычные дни:   0.528
  тяжёлые дни:   0.078


*Микровывод: кол-во таких юзеров ничтожно мало, можно всех оставить. Проблемы в будущем может составить средний reward = 0.078 в тяжёлые дни, но это пока не доказуемо*